# Bulk YouTube Uploader
1. Mount Google Drive
2. Define helper functions
3. Run the uploader

In [ ]:
# @title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title Authentication and Helper Functions
import httplib2
import os
import random
import time
import mimetypes

from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaFileUpload
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials

httplib2.RETRIES = 1
MAX_RETRIES = 10
RETRIABLE_STATUS_CODES = [500, 502, 503, 504]

YOUTUBE_UPLOAD_SCOPE = ["https://www.googleapis.com/auth/youtube.upload"]
YOUTUBE_API_SERVICE_NAME = "youtube"
YOUTUBE_API_VERSION = "v3"

def get_authenticated_service(client_secrets_file):
    creds = None
    if os.path.exists('token.json'):
        try:
            creds = Credentials.from_authorized_user_file('token.json', YOUTUBE_UPLOAD_SCOPE)
        except Exception:
            pass
            
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            try:
                creds.refresh(Request())
            except Exception:
                creds = None
                
        if not creds:
            flow = InstalledAppFlow.from_client_secrets_file(client_secrets_file, YOUTUBE_UPLOAD_SCOPE)
            flow.redirect_uri = 'http://localhost:8080/'
            auth_url, _ = flow.authorization_url(prompt='consent')
            print('Please go to this URL: {}'.format(auth_url))
            print('After authenticating, you will be redirected to localhost:8080. It will fail to load.')
            print('Copy the ENTIRE URL from your browser address bar and paste it below.')
            response = input('Enter the full redirect URL: ')
            flow.fetch_token(authorization_response=response)
            creds = flow.credentials
            
        with open('token.json', 'w') as token:
            token.write(creds.to_json())

    return build(YOUTUBE_API_SERVICE_NAME, YOUTUBE_API_VERSION, credentials=creds)

def initialize_upload(youtube, file_path, title, description, category, tags, privacy_status):
    body=dict(
        snippet=dict(
            title=title,
            description=description,
            tags=tags,
            categoryId=category
        ),
        status=dict(
            privacyStatus=privacy_status
        )
    )

    insert_request = youtube.videos().insert(
        part=",".join(body.keys()),
        body=body,
        media_body=MediaFileUpload(file_path, chunksize=-1, resumable=True)
    )

    return resumable_upload(insert_request)

def resumable_upload(insert_request):
    response = None
    error = None
    retry = 0
    while response is None:
        try:
            print("Uploading file...")
            status, response = insert_request.next_chunk()
            if response is not None:
                if 'id' in response:
                    print("Video id '%s' was successfully uploaded." % response['id'])
                    return True
                else:
                    print("The upload failed with an unexpected response: %s" % response)
                    return False
        except HttpError as e:
            if e.resp.status in RETRIABLE_STATUS_CODES:
                error = "A retriable HTTP error %d occurred:\n%s" % (e.resp.status, e.content)
            else:
                raise
        except Exception as e:
            error = "A retriable error occurred: %s" % e

        if error is not None:
            print(error)
            retry += 1
            if retry > MAX_RETRIES:
                print("No longer attempting to retry.")
                return False

            max_sleep = 2 ** retry
            sleep_seconds = random.random() * max_sleep
            print("Sleeping %f seconds and then retrying..." % sleep_seconds)
            time.sleep(sleep_seconds)
    return False

def find_target_folder(base_path, target_name):
    normalized_target = target_name.replace(" ", "").lower()
    for root, dirs, files in os.walk(base_path):
        for d in dirs:
            if d.replace(" ", "").lower() == normalized_target:
                return os.path.join(root, d)
    return None

def is_video_file(file_path):
    mime_type, _ = mimetypes.guess_type(file_path)
    if mime_type and mime_type.startswith('video'):
        return True
    
    video_exts = ['.mp4', '.mkv', '.avi', '.mov', '.wmv', '.flv', '.webm', '.m4v']
    _, ext = os.path.splitext(file_path)
    return ext.lower() in video_exts


In [ ]:
# @title Bulk Upload Settings
import os

target_folder_name = "Allen Videos" # @param {type:"string"}
client_secrets_file = "/content/client_secrets.json" # @param {type:"string"}
privacy_status = "private" # @param ["public", "private", "unlisted"]
video_category = "22" # @param {type:"string"}
default_tags = "Education" # @param {type:"string"}

if not os.path.exists(client_secrets_file):
    print(f"Error: Client secrets file not found at {client_secrets_file}")
    print("Please upload your client_secrets.json to Colab or Drive.")
else:
    base_drive_path = "/content/drive/MyDrive"
    print(f"Searching for '{target_folder_name}' in {base_drive_path}...")
    
    target_path = find_target_folder(base_drive_path, target_folder_name)
    
    if not target_path:
        print(f"Error: Could not find any folder matching '{target_folder_name}'.")
    else:
        print(f"Found folder at: {target_path}")
        
        print("Authenticating to YouTube...")
        youtube = get_authenticated_service(client_secrets_file)
        
        videos_uploaded = 0
        tags_list = [tag.strip() for tag in default_tags.split(",") if tag.strip()]
        
        for root, dirs, files in os.walk(target_path):
            for file in files:
                file_path = os.path.join(root, file)
                
                if is_video_file(file_path):
                    filename_no_ext = os.path.splitext(file)[0]
                    title = filename_no_ext[:100]
                    description = f"Video uploaded from {target_folder_name}\nOriginal filename: {file}"
                    
                    print(f"\nPreparing to upload: {file_path}")
                    success = initialize_upload(
                        youtube,
                        file_path,
                        title,
                        description,
                        video_category,
                        tags_list,
                        privacy_status
                    )
                    
                    if success:
                        videos_uploaded += 1
                        
        print(f"\nBulk upload completed! Total videos uploaded: {videos_uploaded}")
